In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.color import label2rgb
from scipy import ndimage as ndi
from pathlib import Path
from topostats import io, filters, grains
from topostats.theme import Colormap

In [ ]:
nanoscope = Colormap.nanoscope()


def plot(image, title="", figsize=(10, 10), cmap=nanoscope):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(title)
    ax.imshow(image, cmap=cmap)
    plt.show()

In [ ]:
DATA_FILE = Path("/Users/sylvi/topo_data/roughness-files/data/scan1.spm")
CHANNEL = "Height Sensor"
# Fetch and load image
loadscans = io.LoadScans(img_paths=[DATA_FILE], channel=CHANNEL)
loadscans.get_data()
img_raw = loadscans.image
P_TO_NM = loadscans.pixel_to_nm_scaling
print(f"Pixel to nm scaling factor: {P_TO_NM}")
IMG_SIZE_X_PX = img_raw.shape[1]
IMG_SIZE_Y_PX = img_raw.shape[0]
IMG_SIZE_X_NM = IMG_SIZE_X_PX * P_TO_NM
IMG_SIZE_Y_NM = IMG_SIZE_Y_PX * P_TO_NM
print(f"Image size px: X:{IMG_SIZE_X_PX} px Y:{IMG_SIZE_Y_PX} px")
print(f"Image size nm: X:{IMG_SIZE_X_NM} nm Y:{IMG_SIZE_Y_NM} nm")
plt.imshow(img_raw)

In [ ]:
# Flatten the image
filter_object = filters.Filters(
    image=img_raw,
    filename="filename",
    pixel_to_nm_scaling=P_TO_NM,
    row_alignment_quantile=0.5,
    threshold_method="std_dev",
    threshold_std_dev={"above": 100.0, "below": 100.0},
    gaussian_size=0.1,
    remove_scars={
        "run": False,
        "removal_iterations": 2,
        "threshold_low": 0.250,
        "threshold_high": 0.666,
        "max_scar_width": 4,
        "min_scar_length": 16,
    },
)
filter_object.filter_image()
for image_name, image in filter_object.images.items():
    if image is None:
        continue
    plot(image, title=image_name, figsize=(4, 4))